# bentoml vs hand-rolled fastapi for model serving

bentoml 1.0 came out earlier this year with a nicer api. trying it head-to-head against fastapi for a sklearn model and noting where each wins.

In [ ]:
# fastapi version (typical)
fastapi_app = '''
from fastapi import FastAPI
import joblib
import numpy as np
from pydantic import BaseModel

app = FastAPI()
model = joblib.load('iris.joblib')

class Req(BaseModel):
    features: list[float]

@app.post("/predict")
def predict(r: Req):
    pred = model.predict(np.array([r.features]))
    return {"pred": int(pred[0])}
'''
print(fastapi_app)


In [ ]:
bento_service = '''
import bentoml
from bentoml.io import JSON, NumpyNdarray
import numpy as np

clf = bentoml.sklearn.get("iris_clf:latest").to_runner()
svc = bentoml.Service("iris", runners=[clf])

@svc.api(input=JSON(), output=JSON())
def predict(payload: dict) -> dict:
    arr = np.array([payload["features"]])
    pred = clf.run(arr)
    return {"pred": int(pred[0])}
'''
print(bento_service)


## verdict (after a week)
- bentoml: free batching, runners, packaging into a docker image is `bentoml containerize`. nice for a pure-ml team.
- fastapi: still my choice when the service does more than predict (auth, db calls, multiple endpoints). less magic.

for a pure model-only microservice i would now reach for bentoml first.

In [ ]:
# eval mode for inference
# model.eval()
# with torch.no_grad(): ...
